# 03 Train the scan segmenter (optional)

The package ships a small default model (`scan_segmenter/models/scan_seg_v2.onnx`, ~1.5 MB) used by `appearance_trace="model"`. It is trained on synthetic 1-bit tiles rendered from clean LAS curves and is meant to be universal for B&W raster logs; retrain only if your scans look very different (other grid styles, pen widths, resolution).

This notebook writes the new model to **your own folder** (`OUT_DIR`). It never replaces the bundled default unless you pass `overwrite_default=True` yourself. Needs `torch` (+ `onnx` for export).

The clean LAS curves come from public KGS wells that are not shipped: `fetch_kgs_examples` downloads their LAS files once (checksum-verified) into `~/.cache/pylogdigitizer/kgs`. The three shipped black-and-white examples are kept out of training so they stay test wells. The source of this material is the Kansas Geological Survey website at http://www.kgs.ku.edu/. All Rights Reserved.

This notebook is for the black-and-white scan segmenter. Colour raster logs are separated by ink colour instead; see `04_color_scan_to_las.ipynb`.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "scripts"))
import train_scan_segmenter as tss
from pylogdigitizer.kgs_examples import KGS_EXAMPLES, fetch_kgs_examples

TEST_WELLS = {"USA_A_16", "MOBIL_12-1", "Chitwood_12"}           # shipped examples: kept out of training
TRAIN_WELLS = [w for w, v in KGS_EXAMPLES.items() if w not in TEST_WELLS and any(f[0].endswith(".las") for f in v["files"])]
KGS_ROOT = fetch_kgs_examples(TRAIN_WELLS, files_for={w: [f[0] for f in KGS_EXAMPLES[w]["files"] if f[0].endswith(".las")] for w in TRAIN_WELLS})
LAS_GLOB = str(KGS_ROOT / "*" / "*.las")                         # clean reference LAS curves to render tiles from
OUT_DIR = pathlib.Path.home() / "pylogdigitizer_models" / "my_segmenter"
EXCLUDE = []                                                     # extra wells to keep out (substring of the path)
STEPS = 3000                                                     # ~5 min on Apple MPS, ~35 min on 8 CPU cores


In [ ]:
res = tss.train(LAS_GLOB, OUT_DIR, steps=STEPS, overflow=0.5, exclude=EXCLUDE)
res

Use it (the bundled default is unchanged):

In [ ]:
from pylogdigitizer import workflows as wf
# R = wf.digitize_curve("scan.tif", "track1", scale=(0, 150), backup=(150, 300),
#                       appearance_trace="model", seg_model=str(OUT_DIR / "scan_seg.onnx"))